# 01 · Stage 0: call type

**Purpose:** assign support / external / internal to each call, with a method that would still work on new calls.  
**Decisions implemented:** D1 (and the review in action A18 of `DECISIONS.md`).  
**Inputs → outputs:** `dataset/`, `prompts/stage0_v1.md`, `outputs/stage0_final.csv`, `eval/stage0_review_verdicts.json`, `outputs/bakeoff/metrics.json` → label table, cross-check, review results.

*Run from the project root or from `notebooks/`; nothing is written, and no model is called (replay mode).*

In [1]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image
from ti import nbhelpers as nb
nb.setup()
print("replay mode: no network, no API key, no spend")

replay mode: no network, no API key, no spend


## The decision (D1) in plain words

Task 2 compares support, external and internal calls, but the dataset **has no call-type field**. A title rule ("Support Case…
means support") would work here only because this dataset happens to number its support cases. A real feed would not.
So Stage 0 has three steps:

1. **Certain from metadata:** if nobody outside `@aegiscloud.com` is on the call, it is `internal`. No model, because no judgment is needed.
2. **An LLM for the one real judgment:** with a customer present, is it a *support* call or an *external* (relationship) call? The model
   sees the attendee counts and the **first 8 turns only**. **The title is withheld**, so the title rule can serve as an independent cross-check.
3. **A human reviews the disagreements** between the model and the title rule, plus a sample of agreements.

In [2]:
from ti.load import load_meetings
ms = load_meetings()
internal = [m for m in ms if m.is_internal_by_metadata]
with_customer = [m for m in ms if not m.is_internal_by_metadata]
nb.md(f"**Step 1 (metadata, no model):** {len(internal)} of {len(ms)} calls have no attendee outside `@aegiscloud.com` → `internal`, with certainty. "
      f"The other {len(with_customer)} calls have a customer present and go to the model.")

**Step 1 (metadata, no model):** 30 of 100 calls have no attendee outside `@aegiscloud.com` → `internal`, with certainty. The other 70 calls have a customer present and go to the model.

## Step 2: the prompt

The exact text sent as the system prompt (a version header, then the prompt between the markers):

In [3]:
nb.show_prompt("stage0_v1")

prompts/stage0_v1.md
- Version: stage0_v1
- Decision: D1 in DECISIONS.md
- Input: attendee counts, first 8 turns. The meeting title is withheld (D1 revision note 2). Used only for calls that have an outside attendee; calls with no outside attendee are labeled `internal` by metadata and never reach this prompt.
- Model: set in `.env` as `TI_MODEL_STAGE0` (pinned and recorded in DECISIONS.md when set)
----------------------------------------------------------------------------------------------------
You classify one recorded call at Aegis Cloud Security, a B2B SaaS company, as either "support" or "external". A customer is present on this call.

Definitions:
- support: a customer brings a specific problem or request, and an Aegis support-desk agent handles it. The agent introduces themself as part of Aegis support, often references a case, ticket or the customer's issue, and the call is about resolving that issue.
- external: a scheduled relationship call between an Aegis account manager

What the model receives for one support call and one external call. The assertion checks that the meeting title is **not** in the request.

In [4]:
from ti.stage0 import build_user_message
c = nb.calls()
for label in ("support", "external"):
    mid = c[c.type == label].meeting_id.iloc[0]
    m = next(x for x in ms if x.meeting_id == mid)
    msg = build_user_message(m)
    assert m.title not in msg, "the title must stay hidden from the model"
    print(f"--- a {label} call (title withheld: {m.title not in msg}) ---")
    print(nb.short(msg, 900))
    print()

--- a support call (title withheld: True) ---
attendees: 2 total, 1 from Aegis, 1 from outside Aegis opening: Priya Patel: Hi, this is Priya Patel from Aegis Cloud Security support, am I speaking with Dr. Cheng? Dr. Linda Cheng: Yes, this is Linda Cheng, and I have to say I'm — I'm really frustrated right now, we've been dealing with this API issue for going on three days and it is actively impacting our backup workflows. Priya Patel: I completely understand, Dr. Cheng, and I sincerely apologize for the disruption — I've pulled up your support case, number 3286, and I want to make sure we get to the bottom of this today. Dr. Linda Cheng: Three days, Priya. We're a healthcare organization, we have patient data, we have regulatory obligations — this isn't a minor inconvenience, this is a serious problem. Priya Patel: Absolutely, and I take that very seriously, I do. Can you walk me through what you're seeing on your end? The case not…

--- a external call (title withheld: True) ---
atten

## Results

The frozen labels are in `outputs/stage0_final.csv`. All results below are read from that file.

In [5]:
s0 = pd.read_csv(ROOT / "outputs/stage0_final.csv")
summary = (s0.groupby(["label", "source", "confidence"]).size().rename("calls").reset_index())
nb.table(summary)
nb.md(f"The model answered {int((s0.source == 'llm').sum())} calls; every one came back with confidence "
      f"`{', '.join(sorted(s0[s0.source == 'llm'].confidence.unique()))}`. The model/provider that answered: "
      f"`{', '.join(sorted(s0[s0.source == 'llm'].model.unique()))}`.")

label,source,confidence,calls
external,llm,high,40
internal,metadata,certain,30
support,llm,high,30


The model answered 70 calls; every one came back with confidence `high`. The model/provider that answered: `google/gemini-3.8-flash`.

## The cross-check against the title rule

The title rule says *support* only when the title starts with "Support Case". Where the model and the rule disagree, the disagreement is the interesting part:

In [6]:
ct = pd.crosstab(s0.title_rule_label, s0.label, margins=True)
nb.table(ct.reset_index().rename(columns={"title_rule_label": "title rule \\ model"}))
dis = s0[s0.disagree].copy()
dis["evidence quote"] = dis.evidence_quote.map(lambda q: nb.short(q, 130))
nb.table(dis[["title", "title_rule_label", "label", "confidence", "evidence quote"]])

label,title rule \ model,external,internal,support,All
,external,40,0,3,43
,internal,0,30,0,30
,support,0,0,27,27
,All,40,30,30,100


title,title_rule_label,label,confidence,evidence quote
URGENT: Cobalt Software - Aegis Detect Dashboard Down,external,support,high,"Thank you for calling Aegis Cloud Security support, this is David Kim, how can I help you today?"
URGENT: Blackridge Investments - Complete Loss of Threat Visibility,external,support,high,"Thank you for calling Aegis Cloud Security support, this is Marcus Williams, how can I help you today?"
ESCALATION: Northstar Pharma - Detect Outage Impact on Compliance,external,support,high,"this is Elena Vasquez from Aegis Cloud Security senior support, I have your case pulled up here"


All of them are escalation calls whose title carries no case number ("URGENT:", "ESCALATION:") but which open with a support-desk
agent ("Thank you for calling Aegis Cloud Security support…"). The title rule calls them *external*; the model reads the opening and calls them *support*.

**Was the prompt tuned to get these right?** The plan was to run the prompt *without* any sentence about urgent or escalated
calls, and add one only if these calls came out wrong. Both models in the bake-off labelled them support unaided, so none was added.
Honest caveat: the prompt was written after we had already looked at these calls, so this tests the escalation sentence, not a blind prediction.

In [7]:
bk = nb.read_json("outputs/bakeoff/metrics.json")["stage0"]
rows = []
for model, d in bk.items():
    for x in d["disagreements_run1"]:
        rows.append({"call": nb.short(x["title"], 70), "title rule": x["title_rule"], "model": model, "said": x["model"], "confidence": x["confidence"]})
d = pd.DataFrame(rows)
nb.table(d.pivot_table(index=["call", "title rule"], columns="model", values="said", aggfunc="first").reset_index().fillna("agrees with title rule"))

model,call,title rule,gemini,haiku
,Aegis / Northstar Pharma - Urgent: Detect Outage Impact,external,agrees with title rule,support
,Aegis / Quantum Edge - Renewal Concerns,external,agrees with title rule,support
,ESCALATION: Northstar Pharma - Detect Outage Impact on Compliance,external,support,support
,URGENT: Blackridge Investments - Complete Loss of Threat Visibility,external,support,support
,URGENT: Cobalt Software - Aegis Detect Dashboard Down,external,support,support


## Human review (action A18)

The candidate reviewed 15 calls in the *Stage 0 Review Console* (the console offered a blind mode that hides the model's label and the title-rule label until a verdict is chosen; the candidate cannot confirm it was on, so this is the candidate's own review, not a blind one):
the calls where the model and the title rule disagree, the calls where two different models disagreed with each other, and a seeded sample of agreements.

In [8]:
v = nb.read_json("eval/stage0_review_verdicts.json")
rv = pd.DataFrame(v["items"])
rv["group"] = rv.reasons.map(lambda r: {"disagreement": "model vs title rule disagree", "models": "two models disagreed", "sample": "sampled agreement"}.get(r[0], r[0]))
rv["agrees with model"] = rv.verdict == rv.modelAnswer
rv["agrees with title rule"] = rv.verdict == rv.ruleAnswer
by_group = rv.groupby("group").agg(items=("itemId", "size"), agree_with_model=("agrees with model", "sum"), agree_with_title_rule=("agrees with title rule", "sum")).reset_index()
tot = pd.DataFrame([{"group": "all reviewed", "items": len(rv), "agree_with_model": int(rv["agrees with model"].sum()), "agree_with_title_rule": int(rv["agrees with title rule"].sum())}])
nb.table(pd.concat([by_group, tot], ignore_index=True))
five = rv[rv.group != "sampled agreement"][["title", "group", "verdict", "modelAnswer", "ruleAnswer", "otherAnswer"]].rename(
    columns={"verdict": "your verdict", "modelAnswer": "model", "ruleAnswer": "title rule", "otherAnswer": "other model (Haiku)"})
nb.table(five)

group,items,agree_with_model,agree_with_title_rule
model vs title rule disagree,3,3,0
sampled agreement,10,10,10
two models disagreed,2,2,2
all reviewed,15,15,12


title,group,your verdict,model,title rule,other model (Haiku)
Aegis / Quantum Edge - Renewal Concerns,two models disagreed,external,external,external,support
URGENT: Blackridge Investments - Complete Loss of Threat Visibility,model vs title rule disagree,support,support,external,support
Aegis / Northstar Pharma - Urgent: Detect Outage Impact,two models disagreed,external,external,external,support
ESCALATION: Northstar Pharma - Detect Outage Impact on Compliance,model vs title rule disagree,support,support,external,support
URGENT: Cobalt Software - Aegis Detect Dashboard Down,model vs title rule disagree,support,support,external,support


## What the review does and does not show

In [9]:
n_customer = len(with_customer)
n_dis = int((rv.group == "model vs title rule disagree").sum())
n_agree_all = n_customer - int(s0.disagree.sum())
n_sampled = int((rv.group == "sampled agreement").sum())
n_models = int((rv.group == "two models disagreed").sum())
rule_wrong = int(((rv.group == "model vs title rule disagree") & ~rv["agrees with title rule"]).sum())
ub = nb.upper_bound_zero_errors(n_sampled)
haiku_hits = int(((rv.group != "sampled agreement") & (rv.verdict == rv.otherAnswer)).sum())
n_contested = int((rv.group != "sampled agreement").sum())
nb.md(f"""
- **The title rule is wrong on {rule_wrong} of the {n_customer} customer calls** ({nb.pct(rule_wrong / n_customer)}) by the reviewer's verdicts: the
  {n_dis} escalation calls. The reviewer sided with the model on all {len(rv)} reviewed items.
- **Only part of the agreements were checked.** The model and the title rule agree on {n_agree_all} customer calls; the review covered
  {n_sampled + n_models} of them ({n_sampled} sampled plus {n_models} where two models disagreed). {n_sampled} correct out of {n_sampled} sampled does
  not rule out an error rate of up to about {nb.pct(ub['one_sided_exact'])} (one-sided 95% bound); the cruder *rule of three* gives {nb.pct(ub['rule_of_three'])}.
- **The review is not fully independent.** The reviewer had seen the assistant's reading of the {n_models} contested calls before reviewing them,
  and the prompt was written after the escalation calls were known.
- **The other model.** On the {n_contested} contested calls (not the sample) the reviewer agreed with Haiku on {haiku_hits} and with the chosen model on all {n_contested}.
  This is part of why the decision rule was overridden in D7 (notebook 06).
""")


- **The title rule is wrong on 3 of the 70 customer calls** (4.3%) by the reviewer's verdicts: the
  3 escalation calls. The reviewer sided with the model on all 15 reviewed items.
- **Only part of the agreements were checked.** The model and the title rule agree on 67 customer calls; the review covered
  12 of them (10 sampled plus 2 where two models disagreed). 10 correct out of 10 sampled does
  not rule out an error rate of up to about 25.9% (one-sided 95% bound); the cruder *rule of three* gives 30.0%.
- **The review is not fully independent.** The reviewer had seen the assistant's reading of the 2 contested calls before reviewing them,
  and the prompt was written after the escalation calls were known.
- **The other model.** On the 5 contested calls (not the sample) the reviewer agreed with Haiku on 3 and with the chosen model on all 5.
  This is part of why the decision rule was overridden in D7 (notebook 06).


## Replay, and one honest gap in the code path

The same replay proof as in notebook 00, for Stage 0. **Note:** the frozen Stage 0 labels were produced by the bake-off runner
(`eval/bakeoff.py`), whose cache keys include parameters (`max_tokens`, reasoning effort, seed, run number) that
`src/ti/stage0.py::classify` does not send by default. A fresh live run through `stage0.classify` would therefore write a
*different* cache key. The replay below uses the parameters that actually produced the labels. Aligning the two code paths is
a small clean-up that belongs in a production version; it does not change any result.

In [10]:
nb.table(pd.DataFrame([{"parameter": k, "value": str(val)} for k, val in nb.STAGE0_PARAMS.items()]))
r = nb.replay_check(0)
nb.md(f"Replay of Stage 0: {r['llm_calls']} model calls served from the cache ({r['served_from_cache']}), "
      f"{r['mismatches']} differences from `outputs/stage0_final.csv`.")

parameter,value
max_tokens,3000
provider_order,['google-ai-studio']
run,1
reasoning_effort,low
seed,7


Replay of Stage 0: 70 model calls served from the cache (70), 0 differences from `outputs/stage0_final.csv`.